Notebook ini digunakan untuk evaluasi model pada test set dan konversi model ke ONNX

## Environment setup

#### Install Packages

In [ ]:
# Local laptop only
%pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install torch opencv-python numpy supervision rfdetr gdown "rfdetr[train,loggers]" tqdm Pillow pandas
%pip install torch opencv-python numpy supervision rfdetr gdown "rfdetr[train,loggers]" tqdm Pillow pandas

#### Setting output directory and config

In [ ]:
import os
import torch
import gdown
import zipfile
from rfdetr import RFDETRSmall, RFDETRNano, RFDETRMedium

ROOT_DIR = os.getcwd()
DATASET_DIR = os.path.join(ROOT_DIR, "dataset")
OUTPUT_DIR = os.path.join(ROOT_DIR, "weights")

os.makedirs(DATASET_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Dataset Directory: {DATASET_DIR}")
print(f"Output Directory: {OUTPUT_DIR}")

%cd $ROOT_DIR

#### Download dataset

In [ ]:
zip_path = os.path.join(DATASET_DIR, "dataset.zip")

print("Downloading dataset...")
gdown.download(id=DATASET_FILE_ID, output=zip_path, quiet=False)

print("\nExtracting dataset...")
with zipfile.ZipFile(file=zip_path, mode='r') as zip_ref:
    zip_ref.extractall(DATASET_DIR)

print("Cleaning up zip file...")
os.remove(zip_path)
print("Dataset is ready!")

#### Edit RF-DETR library to allow change the num_queries. !! This is a vibed code !!

Default num_queries pada RF-DETR adalah 300, kode dibawah perlu dieksekusi untuk meningkatkan num_queries menjadi 500

Sumber problem: \
https://github.com/roboflow/rf-detr/issues/674 \
https://github.com/roboflow/rf-detr/issues/674#issuecomment-4190200230

In [ ]:
from pathlib import Path
import importlib
import rfdetr

pkg_root = Path(rfdetr.__file__).resolve().parent

OLD_BLOCK_DETR = '''for name in list(checkpoint["model"].keys()):
        if any(name.endswith(x) for x in query_param_names):
            checkpoint["model"][name] = checkpoint["model"][name][:num_desired_queries]'''

NEW_BLOCK_DETR = '''for name in list(checkpoint["model"].keys()):
        if any(name.endswith(x) for x in query_param_names):
            checkpoint_tensor = checkpoint["model"][name]
            checkpoint_queries = checkpoint_tensor.shape[0]
            if checkpoint_queries > num_desired_queries:
                checkpoint["model"][name] = checkpoint_tensor[:num_desired_queries]
            elif checkpoint_queries < num_desired_queries:
                # Keep checkpoint values for existing slots and preserve model
                # initialization for newly added query slots.
                model_tensor = nn_model.state_dict().get(name)
                if model_tensor is None:
                    del checkpoint["model"][name]
                    continue
                expanded = model_tensor.clone()
                expanded[:checkpoint_queries] = checkpoint_tensor
                checkpoint["model"][name] = expanded'''

OLD_BLOCK_WEIGHTS = '''for name in list(checkpoint["model"].keys()):
        if any(name.endswith(param_name) for param_name in query_param_names):
            checkpoint["model"][name] = checkpoint["model"][name][:num_desired_queries]'''

NEW_BLOCK_WEIGHTS = '''for name in list(checkpoint["model"].keys()):
        if any(name.endswith(param_name) for param_name in query_param_names):
            checkpoint_tensor = checkpoint["model"][name]
            checkpoint_queries = checkpoint_tensor.shape[0]
            if checkpoint_queries > num_desired_queries:
                checkpoint["model"][name] = checkpoint_tensor[:num_desired_queries]
            elif checkpoint_queries < num_desired_queries:
                # Keep checkpoint values for existing slots and preserve model
                # initialization for newly added query slots.
                model_tensor = nn_model.state_dict().get(name)
                if model_tensor is None:
                    del checkpoint["model"][name]
                    continue
                expanded = model_tensor.clone()
                expanded[:checkpoint_queries] = checkpoint_tensor
                checkpoint["model"][name] = expanded'''

OLD_BLOCK_MODULE = '''for name in list(checkpoint["model"].keys()):
            if any(name.endswith(x) for x in query_param_names):
                checkpoint["model"][name] = checkpoint["model"][name][:num_desired_queries]'''

NEW_BLOCK_MODULE = '''for name in list(checkpoint["model"].keys()):
            if any(name.endswith(x) for x in query_param_names):
                checkpoint_tensor = checkpoint["model"][name]
                checkpoint_queries = checkpoint_tensor.shape[0]
                if checkpoint_queries > num_desired_queries:
                    checkpoint["model"][name] = checkpoint_tensor[:num_desired_queries]
                elif checkpoint_queries < num_desired_queries:
                    # Keep checkpoint values for existing slots and preserve model
                    # initialization for newly added query slots.
                    model_tensor = self.model.state_dict().get(name)
                    if model_tensor is None:
                        del checkpoint["model"][name]
                        continue
                    expanded = model_tensor.clone()
                    expanded[:checkpoint_queries] = checkpoint_tensor
                    checkpoint["model"][name] = expanded'''

PATCH_SPECS = [
    ("detr.py", OLD_BLOCK_DETR, NEW_BLOCK_DETR),
    ("models/weights.py", OLD_BLOCK_WEIGHTS, NEW_BLOCK_WEIGHTS),
    ("training/module_model.py", OLD_BLOCK_MODULE, NEW_BLOCK_MODULE),
]

def apply_text_patch(file_path: Path, old_block: str, new_block: str) -> str:
    text = file_path.read_text(encoding="utf-8")

    if new_block in text:
        return "already_patched"

    if old_block not in text:
        return "pattern_not_found"

    backup_path = file_path.with_suffix(file_path.suffix + ".bak")
    if not backup_path.exists():
        backup_path.write_text(text, encoding="utf-8")

    updated = text.replace(old_block, new_block, 1)
    file_path.write_text(updated, encoding="utf-8")
    return "patched"

results = {}
for relative_path, old_block, new_block in PATCH_SPECS:
    target = pkg_root / relative_path
    if not target.exists():
        results[relative_path] = "file_not_found"
        continue
    results[relative_path] = apply_text_patch(target, old_block, new_block)

print("RF-DETR patch summary:")
for relative_path, status in results.items():
    print(f"- {relative_path}: {status}")

failed = [p for p, s in results.items() if s in {"file_not_found", "pattern_not_found"}]
if failed:
    raise RuntimeError(
        "Patch was not fully applied. This may be a different rfdetr version. "
        f"Check these files: {failed}"
    )

import rfdetr.detr
import rfdetr.models.weights
import rfdetr.training.module_model
importlib.reload(rfdetr.detr)
importlib.reload(rfdetr.models.weights)
importlib.reload(rfdetr.training.module_model)

print("Patch applied successfully. You can now run RFDETRSize(num_queries=500, num_select=500).")

In [ ]:
# Check apakah num_queries telah menjadi 500
from rfdetr import RFDETRSmall # ubah model yang diinginkan, misal RFDETRMedium

model = RFDETRSmall(num_queries=500, num_select=500)

# Verify the num_queries now to be 500 instead of the default 300
print(f"Target Configuration Queries: {model.model_config.num_queries}")
print(f"Target Postprocess Select: {model.model_config.num_select}")

if hasattr(model.model, "args") and hasattr(model.model.args, "num_queries"):
    print(f"Underlying PyTorch Architecture Queries: {model.model.args.num_queries}")
else:
    print("Could not find underlying args, but configuration is set.")

if hasattr(model.model.model, "query_feat"):
    print(f"query_feat weight shape: {tuple(model.model.model.query_feat.weight.shape)}")

## Evaluation

#### Clear gpu memory and optimize model for inference

In [ ]:
import os
import gc
import torch
import weakref

def cleanup_gpu_memory(obj=None, verbose: bool = False):
    if not torch.cuda.is_available():
        if verbose:
            print("[INFO] CUDA is not available. No GPU cleanup needed.")
        return

    def get_memory_stats():
        allocated = torch.cuda.memory_allocated()
        reserved = torch.cuda.memory_reserved()
        return allocated, reserved

    torch.cuda.synchronize()

    if verbose:
        alloc, reserv = get_memory_stats()
        print(f"[Before] Allocated: {alloc / 1024**2:.2f} MB | Reserved: {reserv / 1024**2:.2f} MB")

    # Ensure we drop all strong references
    if obj is not None:
        ref = weakref.ref(obj)
        del obj
        if ref() is not None and verbose:
            print("[WARNING] Object not fully garbage collected yet.")

    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

    torch.cuda.synchronize()

    if verbose:
        alloc, reserv = get_memory_stats()
        print(f"[After]  Allocated: {alloc / 1024**2:.2f} MB | Reserved: {reserv / 1024**2:.2f} MB")

PRETRAIN_WEIGHT = os.path.join(OUTPUT_DIR, "checkpoint_best_total.pth")

model = RFDETRMedium(pretrain_weights=PRETRAIN_WEIGHT, num_queries=500, num_select=500)

cleanup_gpu_memory(model, verbose=True)

model.optimize_for_inference()

print(PRETRAIN_WEIGHT)

#### Evaluate model on test set

In [ ]:
from PIL import Image
import supervision as sv
from tqdm import tqdm
# from supervision.metrics import MeanAveragePrecision # jangan pakai ini karena max_dets fixed pada 100, bukan 500
from custom_map_metrics import MeanAveragePrecision # custom map metrics dimana max_dets diubah menjadi 500
from supervision.metrics import Recall, Precision, F1Score

ds = sv.DetectionDataset.from_coco(
    images_directory_path=f"{DATASET_DIR}/test",
    annotations_path=f"{DATASET_DIR}/test/_annotations.coco.json",
)

targets = []
predictions = []

for path, image, annotations in tqdm(ds):
    image = Image.open(path)
    detections = model.predict(image, threshold=0.35) # Ubah threshold biar recall dan precision balance

    det_keys_to_remove = [k for k, v in detections.data.items() if isinstance(v, tuple) or (hasattr(v, '__len__') and len(v) != len(detections))]
    for k in det_keys_to_remove:
        detections.data.pop(k, None)
        
    ann_keys_to_remove = [k for k, v in annotations.data.items() if isinstance(v, tuple) or (hasattr(v, '__len__') and len(v) != len(annotations))]
    for k in ann_keys_to_remove:
        annotations.data.pop(k, None)

    targets.append(annotations)
    predictions.append(detections)

map_metric = MeanAveragePrecision()
map_result = map_metric.update(predictions, targets).compute()
print(map_result)

print("=======================\n")

precision_metric = Precision()
precision_result = precision_metric.update(predictions, targets).compute()
print(precision_result)

print("=======================\n")

recall_metric = Recall()
recall_result = recall_metric.update(predictions, targets).compute()
print(recall_result)

print("=======================\n")

f1_metric = F1Score()
f1_result = f1_metric.update(predictions, targets).compute()
print(f1_result)

In [ ]:
map_result.plot()

In [ ]:
precision_result.plot()

In [ ]:
recall_result.plot()

In [ ]:
f1_result.plot()

## Converting to ONNX for deployment

In [ ]:
!pip install "rfdetr[onnxexport]"
!pip install onnxconverter-common

Convert to model.sim.onnx (FP32)

In [ ]:
from rfdetr import RFDETRSmall, RFDETRNano
import os

PRETRAIN_WEIGHT = os.path.join(OUTPUT_DIR, "checkpoint_best_total.pth")
print(PRETRAIN_WEIGHT)

model = RFDETRNano(pretrain_weights=PRETRAIN_WEIGHT, num_queries=500, num_select=500)

print("Converting to ONNX format...")
model.export(
    output_dir="export_onnx", 
    format="onnx",
    verbose=False
)

Convert FP32 to FP16 (jika diperlukan untuk mengurangi latensi lebih jauh, namun dapat mengurangi akurasi model(?)) \
Kode ini tidak saya jalankan

In [ ]:
import onnx
from onnxconverter_common import float16

input_onnx = "export_onnx/inference_model.sim.onnx"
output_fp16_onnx = "export_onnx/inference_model_fp16.onnx"

print(f"Loading FP32 model from: {input_onnx}")
model = onnx.load(input_onnx)

print("Converting weights to FP16...")
model_fp16 = float16.convert_float_to_float16(model)

onnx.save(model_fp16, output_fp16_onnx)

print("Complete!")